In [1]:
from pathlib import Path
import sys

# Find project root by looking for .env
current = Path.cwd()

PROJECT_ROOT = None

for path in [current] + list(current.parents):
    if (path / ".env").exists():
        PROJECT_ROOT = path
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find project root containing .env"
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("✅ Project root found:")
print(PROJECT_ROOT)

✅ Project root found:
C:\Users\DELL\E-Commerce Operations & Customer Experience Analytics


In [2]:
from dotenv import load_dotenv
import os

env_path = PROJECT_ROOT / ".env"

load_dotenv(env_path, override=True)

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

if not GROQ_API_KEY:
    raise ValueError(
        "GROQ_API_KEY not found in .env"
    )

print("✅ GROQ_API_KEY loaded successfully")
print("Key prefix:", GROQ_API_KEY[:7] + "...")

✅ GROQ_API_KEY loaded successfully
Key prefix: gsk_wcm...


In [3]:
%pip install groq

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
from groq import Groq

client = Groq(api_key=GROQ_API_KEY)

print("✅ Groq client initialized")

✅ Groq client initialized


In [5]:
import json

summary_path = (
    PROJECT_ROOT
    / "outputs"
    / "ai_insights"
    / "eda_summary.json"
)

print("Summary file:", summary_path)
print("Exists:", summary_path.exists())

Summary file: C:\Users\DELL\E-Commerce Operations & Customer Experience Analytics\outputs\ai_insights\eda_summary.json
Exists: True


In [6]:
with open(summary_path, "r", encoding="utf-8") as f:
    eda_summary = json.load(f)

print("✅ EDA summary loaded")
print(json.dumps(eda_summary, indent=2))

✅ EDA summary loaded
{
  "total_revenue_brl": 15421082.85,
  "total_delivered_orders": 96470,
  "average_delivery_days": 12.1,
  "median_delivery_days": 10.0,
  "overall_average_review_score": 4.16,
  "percentage_late_deliveries": 6.8,
  "top_3_categories_by_revenue": [
    "health_beauty",
    "watches_gifts",
    "bed_bath_table"
  ],
  "single_order_customer_percentage": 97.0,
  "median_seller_processing_days": 1.0
}


In [7]:
prompt = f"""
You are a senior E-commerce and Supply Chain Business Analyst.

Analyze the following EDA summary from an Olist Brazilian e-commerce dataset.

Your task is to convert the analytical results into actionable business insights.

EDA SUMMARY:
{json.dumps(eda_summary, indent=2)}

Provide the analysis in the following structure:

1. EXECUTIVE SUMMARY
Give 3-5 important findings from the data.

2. REVENUE INSIGHTS
Discuss important revenue and product-category patterns.

3. CUSTOMER INSIGHTS
Discuss customer purchasing behavior, repeat purchases,
retention and customer concentration.

4. DELIVERY & OPERATIONS INSIGHTS
Identify delivery performance issues, delays,
seller processing performance and operational risks.

5. CUSTOMER EXPERIENCE
Discuss review scores and what they may indicate
about customer satisfaction.

6. KEY BUSINESS RISKS
Identify the most important operational or commercial risks.

7. RECOMMENDATIONS
Give 5 specific and practical recommendations.
Each recommendation should explain:
- What should be done
- Why it matters
- Expected business impact

8. MANAGEMENT TAKEAWAY
End with a short paragraph explaining what management
should prioritize first.

Important rules:
- Use only information supported by the supplied EDA summary.
- Do not invent numbers.
- Do not claim causation unless the data supports it.
- Clearly distinguish observations from recommendations.
- Keep the language professional and suitable for a business presentation.
"""

In [9]:
import json

summary_path = (
    PROJECT_ROOT
    / "outputs"
    / "ai_insights"
    / "eda_summary.json"
)

if not summary_path.exists():
    raise FileNotFoundError(
        f"EDA summary not found:\n{summary_path}\n"
        "Run 03_eda_insights.ipynb first."
    )

with open(summary_path, "r", encoding="utf-8") as f:
    eda_summary = json.load(f)

print("✅ EDA summary loaded")
print(json.dumps(eda_summary, indent=2))

✅ EDA summary loaded
{
  "total_revenue_brl": 15421082.85,
  "total_delivered_orders": 96470,
  "average_delivery_days": 12.1,
  "median_delivery_days": 10.0,
  "overall_average_review_score": 4.16,
  "percentage_late_deliveries": 6.8,
  "top_3_categories_by_revenue": [
    "health_beauty",
    "watches_gifts",
    "bed_bath_table"
  ],
  "single_order_customer_percentage": 97.0,
  "median_seller_processing_days": 1.0
}


In [12]:
import requests

try:
    r = requests.get("https://api.groq.com", timeout=15)
    print("Status:", r.status_code)
    print("✅ Your Python environment can reach Groq")
except Exception as e:
    print("❌ Cannot reach Groq")
    print(type(e).__name__, ":", e)

Status: 200
✅ Your Python environment can reach Groq


In [13]:
from groq import Groq

client = Groq(
    api_key=GROQ_API_KEY,
    timeout=120.0,
    max_retries=2
)

print("✅ Groq client ready")

✅ Groq client ready


In [14]:
test_response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {
            "role": "user",
            "content": "Reply with exactly: Groq connection successful."
        }
    ],
    max_completion_tokens=30,
    include_reasoning=False
)

print(test_response.choices[0].message.content)

Groq connection successful.


In [15]:
response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {
            "role": "system",
            "content": "You are an expert E-commerce and Supply Chain Business Analyst."
        },
        {
            "role": "user",
            "content": prompt
        }
    ],
    temperature=0.2,
    max_completion_tokens=1800,
    include_reasoning=False
)

ai_insights = response.choices[0].message.content

print(ai_insights)

**1. EXECUTIVE SUMMARY**  
- **Revenue & Order Volume**: Total revenue of BRL 15.4 M was generated from 96,470 delivered orders, yielding an average order value of roughly BRL 160.  
- **Delivery Performance**: The average delivery time is 12.1 days, with 6.8 % of orders arriving late. The median delivery time (10 days) is lower than the mean, indicating a right‑skewed distribution with a minority of very long‑haul deliveries.  
- **Customer Behavior**: 97 % of customers made only a single purchase, suggesting low repeat‑purchase rates.  
- **Seller Processing**: Median seller processing time is 1 day, indicating that most sellers prepare orders quickly.  
- **Top Revenue Segments**: Health & beauty, watches & gifts, and bed‑bath‑table categories drive the majority of sales.

---

**2. REVENUE INSIGHTS**  
- **Category Concentration**: The three top categories account for the bulk of revenue, implying that marketing spend and inventory focus should be aligned with these segments.  
- *

In [16]:
from pathlib import Path

output_dir = PROJECT_ROOT / "outputs" / "ai_insights"
output_dir.mkdir(parents=True, exist_ok=True)

output_file = output_dir / "ai_business_insights.txt"

with open(output_file, "w", encoding="utf-8") as f:
    f.write(ai_insights)

print("✅ AI insights saved to:")
print(output_file)

✅ AI insights saved to:
C:\Users\DELL\E-Commerce Operations & Customer Experience Analytics\outputs\ai_insights\ai_business_insights.txt
